**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 24: Tool Calling 학습 데이터 생성

## 🎯 Tool Calling 파인튜닝 데이터 생성

로컬 sLLM에 Tool Calling 능력을 부여하려면 **고품질 학습 데이터**가 필요합니다.
이번 세션에서는 GPT-4o-mini를 사용하여 **합성 데이터(Synthetic Data)**를 생성합니다.

### 데이터 생성 파이프라인

```
도구 정의 → 시나리오 설계 → GPT-4o-mini 생성 → 검증/필터링 → 학습 데이터
```

### Tool Calling 학습 데이터 구조

```json
{
  "messages": [
    {"role": "system", "content": "시스템 프롬프트 + 도구 정의"},
    {"role": "user", "content": "사용자 질문"},
    {"role": "assistant", "content": null, "tool_calls": [...]},
    {"role": "tool", "content": "도구 실행 결과"},
    {"role": "assistant", "content": "최종 응답"}
  ]
}
```

### 학습 목표

- ✅ Tool Calling 학습 데이터 형식 이해
- ✅ 도구 정의 템플릿 작성
- ✅ GPT-4o-mini로 합성 데이터 생성
- ✅ 데이터 검증 및 필터링
- ✅ 데이터 통계 분석

## 1️⃣ 데이터 형식 설계 (system/user/assistant/tool 메시지)

Tool Calling 데이터는 4가지 역할(role)의 메시지로 구성됩니다.

In [ ]:
import os
import json
import random
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

print("✅ OpenAI 클라이언트 초기화 완료")

In [ ]:
# Tool Calling 학습 데이터 형식 설명
print("📊 Tool Calling 학습 데이터 메시지 역할")
print("="*60)

roles = [
    ("system", "시스템 프롬프트 + 사용 가능한 도구 정의", "항상 첫 번째"),
    ("user", "사용자의 질문 또는 요청", "도구 필요/불필요 모두 포함"),
    ("assistant (tool_calls)", "모델이 호출할 도구 선택 및 인자", "content=null, tool_calls 포함"),
    ("tool", "도구 실행 결과 (JSON)", "tool_call에 대한 응답"),
    ("assistant (final)", "도구 결과를 바탕으로 한 최종 응답", "사용자에게 전달할 자연어 응답"),
]

for role, desc, note in roles:
    print(f"\n  🔹 {role}")
    print(f"     설명: {desc}")
    print(f"     비고: {note}")

print("\n📌 도구가 불필요한 경우: system → user → assistant (content만, tool_calls 없음)")

In [ ]:
# 데이터 유형별 예시
print("\n📊 데이터 유형")
print("="*60)

print("""
1️⃣ 단일 도구 호출:
   user: "서울 날씨 알려줘"
   → get_weather(city="서울")
   → 결과 반환
   → 자연어 응답

2️⃣ 멀티 도구 호출:
   user: "서울이랑 부산 날씨 비교해줘"
   → get_weather(city="서울") + get_weather(city="부산")
   → 두 결과 비교하여 응답

3️⃣ 연쇄 도구 호출:
   user: "날씨 확인하고 이메일 보내줘"
   → get_weather → send_email
   → 순차 실행 후 응답

4️⃣ 도구 불필요:
   user: "안녕하세요!"
   → 도구 호출 없이 직접 응답

📌 모든 유형의 데이터를 포함해야 모델이 올바르게 학습합니다!
""")

## 2️⃣ 도구 정의 템플릿 작성

In [ ]:
# 학습에 사용할 도구 정의
TOOLS_DEFINITION = [
    {
        "name": "get_weather",
        "description": "주어진 도시의 현재 날씨 정보를 가져옵니다.",
        "parameters": {
            "city": {"type": "string", "description": "도시 이름"}
        }
    },
    {
        "name": "search_web",
        "description": "웹에서 정보를 검색합니다.",
        "parameters": {
            "query": {"type": "string", "description": "검색 쿼리"}
        }
    },
    {
        "name": "calculate",
        "description": "수학 계산을 수행합니다.",
        "parameters": {
            "expression": {"type": "string", "description": "수학 표현식"}
        }
    },
    {
        "name": "get_stock_price",
        "description": "주식 종목의 현재 가격을 조회합니다.",
        "parameters": {
            "ticker": {"type": "string", "description": "종목 코드"}
        }
    },
    {
        "name": "send_email",
        "description": "이메일을 발송합니다.",
        "parameters": {
            "to": {"type": "string", "description": "수신자"},
            "subject": {"type": "string", "description": "제목"},
            "body": {"type": "string", "description": "본문"}
        }
    }
]

# 시스템 프롬프트 생성
tools_desc = ", ".join([f"{t['name']}({', '.join(t['parameters'].keys())})" for t in TOOLS_DEFINITION])
SYSTEM_PROMPT = f"당신은 도구를 활용할 수 있는 AI 어시스턴트입니다. 사용 가능한 도구: {tools_desc}"

print("✅ 도구 정의 완료")
print(f"📌 총 {len(TOOLS_DEFINITION)}개 도구")
print(f"📌 시스템 프롬프트: {SYSTEM_PROMPT[:100]}...")

for tool in TOOLS_DEFINITION:
    params = ", ".join([f"{k}: {v['type']}" for k, v in tool['parameters'].items()])
    print(f"  🔧 {tool['name']}({params}) - {tool['description']}")

## 3️⃣ GPT-4o-mini로 합성 데이터 생성

GPT-4o-mini에게 다양한 시나리오의 Tool Calling 대화를 생성하도록 요청합니다.

In [ ]:
# 데이터 생성 프롬프트
GENERATION_PROMPT = """당신은 Tool Calling 학습 데이터를 생성하는 전문가입니다.

아래 도구 목록을 사용하여, 한국어 Tool Calling 대화 데이터를 JSON 형식으로 생성해주세요.

사용 가능한 도구:
{tools_desc}

데이터 형식:
{{
  "messages": [
    {{"role": "system", "content": "시스템 프롬프트"}},
    {{"role": "user", "content": "사용자 질문"}},
    {{"role": "assistant", "content": null, "tool_calls": [{{"type": "function", "function": {{"name": "함수명", "arguments": "{{\\"param\\": \\"value\\"}}"}}}}]}},
    {{"role": "tool", "content": "{{\\"key\\": \\"value\\"}}"}},
    {{"role": "assistant", "content": "최종 응답"}}
  ]
}}

요청 유형: {scenario_type}

규칙:
1. 한국어로 자연스러운 대화를 생성하세요
2. 도구 인자는 JSON 문자열로 제공하세요
3. 도구 결과는 현실적인 JSON을 생성하세요
4. 최종 응답은 도구 결과를 바탕으로 친절하게 작성하세요
5. JSON만 출력하세요 (추가 설명 없이)
"""

# 시나리오 유형
SCENARIO_TYPES = [
    "단일 도구 호출 (날씨 조회)",
    "단일 도구 호출 (계산)",
    "단일 도구 호출 (검색)",
    "단일 도구 호출 (주가 조회)",
    "단일 도구 호출 (이메일 발송)",
    "멀티 도구 호출 (날씨 2개 도시 비교)",
    "멀티 도구 호출 (주가 2개 종목 비교)",
    "도구 불필요 (일반 인사)",
    "도구 불필요 (일반 지식 질문)",
    "도구 불필요 (감사/인사)"
]

print(f"✅ {len(SCENARIO_TYPES)}개 시나리오 유형 정의")
for i, s in enumerate(SCENARIO_TYPES):
    print(f"  {i+1}. {s}")

In [ ]:
# 데이터 생성 함수
def generate_tool_calling_data(scenario_type, num_samples=1):
    """GPT-4o-mini로 Tool Calling 학습 데이터 생성"""
    tools_desc_str = "\n".join([
        f"- {t['name']}({', '.join([f'{k}: {v["type"]}' for k, v in t['parameters'].items()])}): {t['description']}"
        for t in TOOLS_DEFINITION
    ])
    
    prompt = GENERATION_PROMPT.format(
        tools_desc=tools_desc_str,
        scenario_type=scenario_type
    )
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.8,
            max_tokens=1000,
            response_format={"type": "json_object"}
        )
        
        content = response.choices[0].message.content
        data = json.loads(content)
        
        # 시스템 프롬프트 통일
        if "messages" in data and len(data["messages"]) > 0:
            data["messages"][0] = {"role": "system", "content": SYSTEM_PROMPT}
        
        return data
    except Exception as e:
        print(f"  ⚠️ 생성 실패 ({scenario_type}): {e}")
        return None

print("✅ 데이터 생성 함수 정의 완료")

In [ ]:
# 데이터 생성 실행
print("🚀 Tool Calling 학습 데이터 생성 시작")
print("="*60)

generated_data = []
num_per_scenario = 2  # 각 시나리오당 2개씩 생성

for scenario in SCENARIO_TYPES:
    print(f"\n⏳ 생성 중: {scenario}")
    for i in range(num_per_scenario):
        data = generate_tool_calling_data(scenario)
        if data and "messages" in data:
            generated_data.append(data)
            print(f"  ✅ {i+1}/{num_per_scenario} 생성 완료")
        else:
            print(f"  ⚠️ {i+1}/{num_per_scenario} 생성 실패")

print(f"\n{'='*60}")
print(f"✅ 총 {len(generated_data)}개 데이터 생성 완료")

In [ ]:
# 생성된 데이터 미리보기
print("📊 생성된 데이터 미리보기")
print("="*60)

for i, data in enumerate(generated_data[:3]):
    print(f"\n🔹 샘플 {i+1}:")
    for msg in data["messages"]:
        role = msg["role"]
        content = msg.get("content", "")
        tool_calls = msg.get("tool_calls", None)
        
        if tool_calls:
            for tc in tool_calls:
                func = tc.get("function", {})
                print(f"  [{role}] → {func.get('name', '?')}({func.get('arguments', '?')})")
        else:
            content_preview = str(content)[:80] if content else "(null)"
            print(f"  [{role}] {content_preview}")
    print()

## 4️⃣ 데이터 검증 및 필터링

생성된 데이터의 품질을 검증하고 불량 데이터를 필터링합니다.

In [ ]:
# 데이터 검증 함수
def validate_tool_calling_data(data):
    """Tool Calling 학습 데이터 유효성 검증"""
    errors = []
    
    # 기본 구조 확인
    if "messages" not in data:
        return ["'messages' 키 없음"]
    
    messages = data["messages"]
    if len(messages) < 2:
        return ["메시지가 2개 미만"]
    
    # system 메시지 확인
    if messages[0]["role"] != "system":
        errors.append("첫 번째 메시지가 system이 아님")
    
    # user 메시지 확인
    has_user = any(m["role"] == "user" for m in messages)
    if not has_user:
        errors.append("user 메시지 없음")
    
    # assistant 메시지 확인
    has_assistant = any(m["role"] == "assistant" for m in messages)
    if not has_assistant:
        errors.append("assistant 메시지 없음")
    
    # tool_calls가 있으면 tool 응답이 있는지 확인
    for i, msg in enumerate(messages):
        if msg.get("tool_calls"):
            # 다음 메시지가 tool인지 확인
            if i + 1 < len(messages) and messages[i + 1]["role"] != "tool":
                errors.append("tool_calls 후 tool 응답 없음")
            
            # tool_calls 형식 확인
            for tc in msg["tool_calls"]:
                if "function" not in tc:
                    errors.append("tool_call에 function 없음")
                elif "name" not in tc["function"]:
                    errors.append("function에 name 없음")
    
    return errors

# 검증 실행
print("📊 데이터 검증 결과")
print("="*60)

valid_data = []
invalid_count = 0

for i, data in enumerate(generated_data):
    errors = validate_tool_calling_data(data)
    if errors:
        print(f"  ⚠️ 샘플 {i+1}: {errors}")
        invalid_count += 1
    else:
        valid_data.append(data)

print(f"\n✅ 유효 데이터: {len(valid_data)}개")
print(f"⚠️ 무효 데이터: {invalid_count}개")
print(f"📌 유효율: {len(valid_data)/max(len(generated_data),1)*100:.1f}%")

## 5️⃣ 샘플 데이터 확인

기존에 준비된 샘플 데이터를 확인합니다.

In [ ]:
# 기존 샘플 데이터 로드
sample_path = "../data/samples/tool_calling_sample.json"

with open(sample_path, "r", encoding="utf-8") as f:
    sample_data = json.load(f)

print(f"✅ 샘플 데이터 로드 완료: {len(sample_data)}개")
print(f"📌 경로: {sample_path}")

# 샘플 데이터 미리보기
print(f"\n--- 샘플 데이터 미리보기 (처음 3개) ---")
for i, item in enumerate(sample_data[:3]):
    print(f"\n🔹 샘플 {i+1}:")
    for msg in item["messages"]:
        role = msg["role"]
        content = msg.get("content", "")
        tool_calls = msg.get("tool_calls", None)
        
        if tool_calls:
            for tc in tool_calls:
                func = tc.get("function", {})
                print(f"  [{role}] → {func.get('name', '?')}({func.get('arguments', '?')[:50]})")
        else:
            preview = str(content)[:80] if content else "(null)"
            print(f"  [{role}] {preview}")

In [ ]:
# 생성 데이터와 샘플 데이터 합치기
all_data = sample_data + valid_data
print(f"📊 전체 데이터 구성")
print(f"  📄 샘플 데이터: {len(sample_data)}개")
print(f"  📄 생성 데이터: {len(valid_data)}개")
print(f"  📄 합계: {len(all_data)}개")

## 6️⃣ 데이터 통계 분석

In [ ]:
# 데이터 통계 분석
print("📊 Tool Calling 데이터 통계 분석")
print("="*60)

# 도구 호출 유형별 분류
tool_call_count = 0
no_tool_count = 0
multi_tool_count = 0
tool_usage = {}

for item in all_data:
    has_tool_call = False
    call_count = 0
    
    for msg in item["messages"]:
        if msg.get("tool_calls"):
            has_tool_call = True
            for tc in msg["tool_calls"]:
                call_count += 1
                func_name = tc.get("function", {}).get("name", "unknown")
                tool_usage[func_name] = tool_usage.get(func_name, 0) + 1
    
    if has_tool_call:
        tool_call_count += 1
        if call_count > 1:
            multi_tool_count += 1
    else:
        no_tool_count += 1

print(f"\n🔹 전체 데이터: {len(all_data)}개")
print(f"   - 도구 호출 있음: {tool_call_count}개 ({tool_call_count/len(all_data)*100:.0f}%)")
print(f"   - 도구 호출 없음: {no_tool_count}개 ({no_tool_count/len(all_data)*100:.0f}%)")
print(f"   - 멀티 도구 호출: {multi_tool_count}개")

print(f"\n🔹 도구별 사용 빈도:")
for tool_name, count in sorted(tool_usage.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f"   {tool_name:<20} {count:>3}회 {bar}")

# 메시지 길이 통계
msg_lengths = []
for item in all_data:
    total_len = sum(len(str(msg.get("content", ""))) for msg in item["messages"])
    msg_lengths.append(total_len)

print(f"\n🔹 메시지 길이 통계:")
print(f"   최소: {min(msg_lengths)}자")
print(f"   최대: {max(msg_lengths)}자")
print(f"   평균: {sum(msg_lengths)/len(msg_lengths):.0f}자")

In [ ]:
# 데이터 저장
output_path = "../data/samples/tool_calling_training_data.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

file_size = os.path.getsize(output_path)
print(f"✅ 학습 데이터 저장 완료")
print(f"📌 경로: {output_path}")
print(f"📌 크기: {file_size/1024:.1f} KB")
print(f"📌 샘플 수: {len(all_data)}")

In [ ]:
# 데이터 품질 체크리스트
print("\n📋 데이터 품질 체크리스트")
print("="*60)
print(f"""
✅ 다양한 도구 포함: {len(tool_usage)}개 도구
✅ 도구 호출/미호출 균형: {tool_call_count}:{no_tool_count}
✅ 멀티 도구 호출 포함: {multi_tool_count}개
✅ 한국어 자연스러움: 확인 필요 (수동 검토)
✅ 도구 인자 정확성: 검증 완료
✅ 응답 품질: 확인 필요 (수동 검토)

📌 실제 프로젝트에서는:
   - 최소 500~1000개 이상의 데이터 권장
   - 각 도구별 균형 있는 분포
   - 사람이 직접 검수하는 단계 필요
   - 에지 케이스(오타, 모호한 질문) 포함 권장
""")

## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

| 항목 | 내용 |
|------|------|
| 데이터 형식 | system/user/assistant(tool_calls)/tool/assistant(final) |
| 도구 정의 | JSON Schema로 함수명, 설명, 파라미터 정의 |
| 합성 데이터 | GPT-4o-mini로 다양한 시나리오 자동 생성 |
| 검증/필터링 | 구조 검증, 형식 확인, 품질 평가 |

### 핵심 포인트

- 🎯 **데이터 다양성이 핵심** - 단일/멀티/미호출 모두 포함
- 🎯 **도구 정의의 일관성** - 시스템 프롬프트에 도구 정보 통일
- 🎯 **검증 필수** - 자동 검증 + 수동 검토 병행
- 🎯 **실제 프로젝트는 500+ 데이터** 필요
- 🎯 **GPT-4o-mini 비용 효율적** - 대량 생성에 적합

### 다음 단계

- ➡️ **Notebook 24**: 생성된 데이터로 Qwen2.5-3B를 Tool Calling 파인튜닝!